# Typeclasses

In Gallowglass, typeclasses describe a set of operations that can be implemented for different types. The compiler dispatches to the right implementation based on inferred types — same syntax in the caller, different code at runtime.

This notebook covers the built-in `Eq` (equality), `Ord` (ordering), and `Show` (rendering) classes, shows how to define instances for your own types, and explains the one mechanical wrinkle the bootstrap currently has around dispatch.

This lesson assumes you've worked through `01-hello-gallowglass.ipynb` — declarations, types, ADTs, pattern matching.

## Built-in: `Eq`

Bring `Eq` and the method `eq` into scope from `Core.Nat`:

In [1]:
use Core.Nat unqualified { Eq, eq }

use Core.Nat

To call a class method, define a **constrained wrapper**. In the bootstrap, class-method dispatch only fires when the call goes through a `let` whose type carries the constraint (`Eq a =>`); bare calls like `eq 5 5` surface as `unbound variable` codegen errors. The wrapper makes the dispatch fire:

In [2]:
let same : ∀ a. Eq a => a → a → Bool = λ x y → eq x y

same : ∀ a. a → a → Bool

In [3]:
same 5 5

True

In [4]:
same 5 7

False

## Built-in: `Show`

The kernel's type-driven renderer already gives nice output for `42` or `True` without going through `Show`. But if you want to call `show` explicitly — say, to embed a rendered value in a string — define a wrapper:

In [5]:
use Core.Text { Show, show }

use Core.Text

In [6]:
let display : ∀ a. Show a => a → Text = λ x → show x

display : ∀ a. a → Text

In [7]:
display 42

"42"

`Show` instances exist in the prelude for `Nat`, `Bool`, `Text`, `Pair`, `Option`, `List`, and `Result`. The instances for compound types depend on a `Show` instance for their element type — that's the `Show a => Show List` shape.

## Custom instances

Define your own type and give it an `Eq` instance. The instance body provides each method:

In [8]:
type Coin = | Heads | Tails

type Coin = Heads | Tails

In [9]:
instance Eq Coin {
  eq = λ x y → match x {
    | Heads → match y { | Heads → True  | Tails → False }
    | Tails → match y { | Heads → False | Tails → True  }
  }
}

Now `same` works on `Coin` — the typechecker selects the instance we just defined:

In [10]:
same Heads Heads

True

In [11]:
same Heads Tails

False

## Built-in: `Ord`

`Ord` extends `Eq` (it has `Eq` as a superclass). Bring it into scope and define a `less` wrapper:

In [12]:
use Core.Nat unqualified { Ord, lt }

use Core.Nat

In [13]:
let less : ∀ a. Ord a => a → a → Bool = λ x y → lt x y

less : ∀ a. a → a → Bool

In [14]:
less 3 5

True

In [15]:
less 5 3

False

## What's next

The next notebook is about Glass IR — the typed intermediate representation the compiler emits, where every binding gets a BLAKE3-256 pin hash that content-addresses it.